# Milestone 4 lab, notebook 6 of 7: Tests on the tables

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| **6. Tests on the tables** (you are here) | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

Every test so far in this project checked **code**, on data made for the test. Nothing looked at the tables after a real run. If the order system sent something strange tomorrow, the pipeline would load it, add it up and report it.

This notebook puts tests on the **tables**, and then sends them bad data on purpose.

In [1]:
from psycopg.types.json import Jsonb

from lab_helpers import MILESTONE_DIR, dbt, lab_warehouse, load_only, money, show_doc, table

con = lab_warehouse(fresh=True)
load_only(con, "2026-06-08", "2026-06-09")
dbt("build", show=False)
print("\nThe models are built and tested:", dbt.passed)

Landed 226 raw rows for the morning of 2026-06-08.
Landed 102 raw rows for the morning of 2026-06-09.



The models are built and tested: True


## Step 1: A test is a query that looks for trouble

Write one by hand. The rule: *one order is one row*. Look for order ids that appear more than once.

In [2]:
table(con, """
    SELECT order_id, count(*) AS times
    FROM core.orders
    GROUP BY order_id
    HAVING count(*) > 1
""")

,order_id,times


No rows. That is a **pass**: the query looked for rows that break the rule, and found none.

A **table test** is exactly this. It is a query that returns the rows that are wrong. Zero rows means the table is right.

## Step 2: Four tests you get by naming them

Writing that query for every key would be dull. dbt has four tests built in. You only name the column and the test.

In [3]:
print((MILESTONE_DIR / "dbt_project" / "models" / "core" / "core.yml").read_text()[:1500])

version: 2

models:
  - name: orders
    description: One row per order, with its latest state. The grain is one order.
    columns:
      - name: order_id
        description: The key. One row per order.
        tests: [unique, not_null]
      - name: status
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: [placed, packed, out_for_delivery, delivered, cancelled]
      - name: city
        tests:
          - not_null
          - relationships:
              arguments:
                to: ref('cities')
                field: city
      - name: city_name
        tests: [not_null]
      - name: order_date
        tests: [not_null]
      - name: total_cents
        tests: [not_null]
      - name: refunded_cents
        tests: [not_null]

  - name: order_items
    description: One row per item line. The grain is one line of one order.
    columns:
      - name: order_id
        tests:
          - not_null
          - relations

| Test | The question it asks | Arden's rule |
|---|---|---|
| `unique` | Does any value appear twice? | One order is one row |
| `not_null` | Is any value missing? | Every order has a status, a city, a total |
| `accepted_values` | Is any value outside the list? | An order has one of five statuses |
| `relationships` | Does any value point at nothing? | Every item belongs to an order |

Run the tests of the orders table.

In [4]:
dbt("test", "--select", "orders")

1 of 14 PASS accepted_values_orders_status__placed__packed__out_for_delivery__delivered__cancelled  [PASS]
2 of 14 PASS assert_money_is_never_negative .................................... [PASS]
3 of 14 PASS assert_order_total_equals_its_items ............................... [PASS]
4 of 14 PASS assert_report_equals_its_orders ................................... [PASS]
5 of 14 PASS not_null_orders_city .............................................. [PASS]
6 of 14 PASS not_null_orders_city_name ......................................... [PASS]
7 of 14 PASS not_null_orders_order_date ........................................ [PASS]
8 of 14 PASS not_null_orders_order_id .......................................... [PASS]
9 of 14 PASS not_null_orders_refunded_cents .................................... [PASS]
10 of 14 PASS not_null_orders_status ........................................... [PASS]
11 of 14 PASS not_null_orders_total_cents ...................................... [PASS]
12 of 14 PASS

## Step 3: Tests written as SQL

Some rules need more than one column, or more than one table. Those are written as a `SELECT`, in the `tests` folder. This one is the most important test in the project.

In [5]:
print((MILESTONE_DIR / "dbt_project" / "tests" / "assert_report_equals_its_orders.sql").read_text())

-- The report's totals must equal the sum of its orders.
-- A test passes when its query returns no rows. Each row returned is a day that is wrong.

with from_report as (

    select order_date, sum(orders) as orders, sum(revenue_cents) as revenue_cents
    from {{ ref('daily_orders') }}
    group by order_date

),

from_orders as (

    select
        order_date,
        count(*) filter (where status <> 'cancelled')                        as orders,
        coalesce(sum(net_cents) filter (where status <> 'cancelled'), 0)     as revenue_cents
    from {{ ref('orders') }}
    where not is_deleted
    group by order_date

)

select
    coalesce(r.order_date, o.order_date) as order_date,
    r.orders        as report_orders,
    o.orders        as true_orders,
    r.revenue_cents as report_cents,
    o.revenue_cents as true_cents
from from_report as r
full join from_orders as o using (order_date)
where r.orders is distinct from o.orders
   or r.revenue_cents is distinct from o.revenue_cen

It adds the days up twice, once from the report and once from the orders, and returns every day where the two differ. It is the answer key of milestone 1, turned into a check that runs every morning.

## Step 4: Break the data

Now be the order system on a bad day. Land a new version of order 1130 with a status nobody has ever seen.

In [6]:
def land_changed(order_id, **changes):
    """Land a new version of an order, as if the source had sent it."""
    payload = con.execute(
        "SELECT payload FROM raw.orders_landing WHERE (payload->>'order_id')::bigint = %s ORDER BY landing_id DESC LIMIT 1",
        [order_id],
    ).fetchone()[0]
    payload.update(changes, updated_at="2026-06-30T00:00:00Z")
    con.execute("INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES (%s, 'bad_data', 1, now())", [Jsonb(payload)])

def report_for_friday():
    row = con.execute("SELECT sum(orders), sum(revenue_cents) FROM marts.daily_orders WHERE order_date = '2026-06-05'").fetchone()
    return f"{row[0]} orders, {money(int(row[1]))}"

print("Friday 5 June in the report, before:", report_for_friday())
land_changed(1130, status="teleported")

Friday 5 June in the report, before: 37 orders, $971.04


In [7]:
dbt("build")
print("\nDid the build pass?", dbt.passed)

1 of 38 OK created sql view model staging.stg_order_deletes .................... [CREATE VIEW]
2 of 38 OK loaded seed file core.cities ........................................ [INSERT 3]
3 of 38 PASS source_not_null_raw_orders_landing_landing_id ..................... [PASS]
4 of 38 PASS source_not_null_raw_orders_landing_payload ........................ [PASS]
5 of 38 PASS source_unique_raw_orders_landing_landing_id ....................... [PASS]
6 of 38 PASS not_null_stg_order_deletes_order_id ............................... [PASS]
7 of 38 PASS unique_stg_order_deletes_order_id ................................. [PASS]
8 of 38 PASS not_null_cities_city .............................................. [PASS]
9 of 38 PASS unique_cities_city ................................................ [PASS]
10 of 38 OK created sql view model staging.stg_orders .......................... [CREATE VIEW]
11 of 38 PASS not_null_stg_orders_order_id ..................................... [PASS]
12 of 38 PASS 

Find three lines in that output.

1. `FAIL 1 accepted_values_orders_status...`: the test found one row with a status outside the list.
2. `SKIP relation marts.daily_orders`: the report was **not rebuilt**. It depends on `orders`, and a test on `orders` failed.
3. The summary at the end: `ERROR=1`, and some `SKIP`.

The test did not just complain. It stopped the bad row from reaching the manager.

In [8]:
print("Friday 5 June in the report, after the failed run:", report_for_friday())

Friday 5 June in the report, after the failed run: 37 orders, $971.04


The report still shows its last good numbers. An old right number is better than a new wrong one.

Nothing was lost either. The strange row is in the raw table, where it can be studied:

In [9]:
table(con, "SELECT landing_id, run_id, payload->>'order_id' AS order_id, payload->>'status' AS status FROM raw.orders_landing WHERE run_id = 'bad_data'")

,landing_id,run_id,order_id,status
0,329,bad_data,1130,teleported


## Step 5: Two more kinds of bad data

Take the bad row out, and try a total that is too high for its items. Then a negative one.

In [10]:
def try_bad(**changes):
    con.execute("DELETE FROM raw.orders_landing WHERE run_id = 'bad_data'")
    land_changed(1130, **changes)
    dbt("build", show=False)
    return "the run passed ❌" if dbt.passed else "the run FAILED ✅"

print("A total of $99.99 that its items do not add up to:", try_bad(total_cents=9999))
print("A total of -$5.00:                                 ", try_bad(total_cents=-500))
print("A city nobody knows, 'XXX':                        ", try_bad(city="XXX"))

A total of $99.99 that its items do not add up to: the run FAILED ✅


A total of -$5.00:                                  the run FAILED ✅


A city nobody knows, 'XXX':                         the run FAILED ✅


Each one is caught by a different test. The last one is the test promised in notebook 3: the left join kept the order with an empty city name, and a test found it.

Now the source is fixed. Remove the bad row and run again.

In [11]:
con.execute("DELETE FROM raw.orders_landing WHERE run_id = 'bad_data'")
dbt("build", show=False)
print("Did the build pass?", dbt.passed, "  Friday 5 June:", report_for_friday())

Did the build pass? True   Friday 5 June: 37 orders, $971.04


## Step 6: Is the data too old?

One kind of wrong data passes every test above: data that is **right, but old**. If the load silently stops, the tables stay perfectly valid, and the manager reads last week's numbers.

So there is one more check. It looks at `loaded_at`, the load time on every raw row, and asks: is the newest row recent enough?

In [12]:
print((MILESTONE_DIR / "dbt_project" / "models" / "sources.yml").read_text())

version: 2

sources:
  - name: raw
    description: Data exactly as it arrived. The Python load writes it, by appending. dbt only reads it.
    schema: raw
    loaded_at_field: loaded_at
    freshness:                       # is the newest raw row recent enough?
      warn_after: {count: 24, period: hour}
      error_after: {count: 48, period: hour}
    tables:
      - name: orders_landing
        description: One row for every order the API sent, every time it sent it.
        columns:
          - name: landing_id
            tests: [unique, not_null]
          - name: payload
            tests: [not_null]
      - name: order_deletes
        description: One row for every delete read from the order system's change feed.
        freshness: null              # a day without deletes is normal



In [13]:
dbt("source", "freshness")

1 of 1 PASS freshness of raw.orders_landing .................................... [PASS]
Done.


The rows were loaded a moment ago, so it passes. Now pretend the load has not run for three days.

In [14]:
con.execute("UPDATE raw.orders_landing SET loaded_at = loaded_at - interval '3 days'")
dbt("source", "freshness")
print("\nDid the freshness check pass?", dbt.passed)

1 of 1 ERROR STALE freshness of raw.orders_landing ............................. [ERROR STALE]
[ERROR]: in source orders_landing (models/sources.yml)
Status: error
Done.

Did the freshness check pass? False


`ERROR STALE`. The newest row is more than 48 hours old.

Notice which time this uses. It is **load time**, not event time. In milestone 1 you learned never to use load time for a business number. For "did my pipeline run?", load time is exactly the right question.

In [15]:
con.close()
show_doc("LLD.md", "6. Table tests")

**From `docs/LLD.md`**

> ## 6. Table tests
> 
> A test is a query that looks for **rows that break a rule**. It passes when it finds none.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Table test** | A check on the data in a table. It is a query that returns the rows that break a rule, and passes when it returns none | 31 tests run on every build |
| **Freshness** | How new the data in the warehouse is. The check: is the newest loaded row recent enough? | Warn after 24 hours, error after 48 |

You met freshness as an idea in milestone 1. Here it is a check that can fail.

## What you now know

- A table test is a query for rows that are wrong. No rows means the table is right.
- `unique`, `not_null`, `accepted_values` and `relationships` are named in a YAML file. Other rules are written as SQL.
- When a test fails, the models after it are skipped. The report keeps its last good numbers.
- Freshness uses load time to ask whether the pipeline ran.

**Say it in an interview:** "I test the tables, not only the code. Unique and not-null on keys, accepted values on status, a relationship from items to orders, and a reconciliation that the report equals the sum of its orders. I ran the experiment: when the source sent an unknown status, the build failed, the mart was skipped, and the report kept yesterday's numbers."

Next: [notebook 7](07_the_production_version.ipynb) puts the load and dbt together as one program.